# Agentic Loop

In this lesson, we'll clean it up by implementing an agentic loop that automatically handles tool calls until the agent is done.

Every agent framework runs on the agentic loop. It's also called the tool call loop or function call loop.



In [4]:
from openai import OpenAI

openai_client = OpenAI()


In [6]:
from minsearch import AppendableIndex

In [8]:
# Import json for parsing tool call arguments. Still need our docs for information.
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex


reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [9]:
# Define the search function, with the search tool definition:

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

## Handling Tool Calls Manually

Let's see what we had before. Define instructions and question:

In [ ]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

question = "How do I create a dahsbord in Evidently?" # The qustion has a typo

# Create the initial message history:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

# Make the initial API call:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

In [ ]:
# Extend the message history with the response:

message_history.extend(response.output)

message_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different\nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard"}', call_id='call_pDJgwfVBNc26a7IAgtwNUYmb', name='search', type='function_call', id='fc_05562f2c00983a0f006afb0ab511e487d29239869e8fe9b40f', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"dashboard creation steps"}', call_id='call_hd6LbmYj0eorEv5wuM2XPmzU', name='search', type='function_call', id=

## The make_call Function
Instead of manually handling each tool call, let's create a generic function that can handle any tool:



In [14]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

# This function: - Parses the tool call arguments - Checks which tool to call - Executes the tool - Returns the result in the format expected by OpenAI

# Note how it returns an error message for unknown tools instead of raising an exception. This lets the agent self-correct if it chooses the wrong function.


In [15]:
# The agent made three tool calls in one iteration. Now we need to execute them:

# Loop through the messages and execute the tool calls:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)

# Now we can make another API call with the updated message history:

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

print(response.output_text)


executing search({"query":"create dashboard"})...
executing search({"query":"dashboard creation steps"})...
executing search({"query":"dashboard features"})...
To create a dashboard in Evidently, follow these steps:

### Step 1: Set Up the Project
Before creating a dashboard, ensure you have connected to Evidently Cloud and created a project. This is essential as each project has its own dashboard.

### Step 2: Dashboard Overview
- **Dashboards** provide a clear view of your AI application's performance and can track evaluation results across experiments or production quality over time.
- Initially, the dashboard will be empty, which you can populate by running evaluations and saving reports.

### Step 3: Adding Tabs
1. **Enter "Edit" Mode**: In your dashboard, switch to "Edit" mode (top right corner).
2. **Add a Tab**:
   - Click the plus sign for "Add Tab" on the left.
   - You can create a custom tab by selecting "empty" and entering a name.

### Step 4: Adding Panels
You can add va

In [ ]:
len(message_history)
# The message history now has 8 messages, including the initial question, the three tool calls, and the final response.

# The agent now has the answer and can continue the conversation.

# This is the agentic loop. The agent keeps making tool calls until it has enough information to answer the question.

# This is a simple example, but it can be extended to handle more complex tasks and use multiple tools.

# This is the foundation of agentic RAG.


8

# The Agentic Loop 

Instead of manually handling each step, we can use a loop that continues until the agent doesn't want to make any more tool calls.

Update the instructions to test multi-iteration behavior:



In [17]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [19]:
# Now define the agentic loop:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To provide a thorough answer on how to create a dashboard, I will first search for specific documentation about dashboard creation. This will help me gather relevant steps and guidelines specific to that process. 

Let's start with this initial search.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: The search provided useful information regarding the creation of dashboards, outlining essential steps and methods to add panels and manage tabs. Based on the initial results, there are two primary methods for adding panels: using the Python API and through the user interface. 

To expand on how to create a dashboard in the user interface, I will perform two additional searches—one to delve deeper into the user interface process and another to explore the overall management of dashboards. This will ensure comprehensive coverage of the dashboard creation process. 

Let's proceed with the next two searches.
executing sea

In [18]:
# ^ this loop will continue until the agent doesn't want to make any more tool calls.

# The loop: 
# 1. Sends the entire message history to the LLM 
# 2. Checks each response message 
# 3. If it's a function call, executes it and appends the result 
# 4. If it's a message (final answer), prints it 
# 5. Breaks when no more function calls are made

# When you run this, the agent makes 3 iterations: 
# 1. First iteration: one search for "create dashboard" 
# 2. Second iteration: analyzed results and made two more searches 
# 3. Third iteration: wrote the answer

# Adding More Tools

In [20]:
# With this approach, adding new tools is easy. 
# Let's add an add_entry function that can add new documentation to the index:


# Add a new tool to the index:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [21]:
# Update make_call to handle the new tool:

def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [22]:
# We can now use multiple tools together. Let's update the instructions to use both tools.

# Test with both tools available. Define the tools list and run the loop:

message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To provide a comprehensive answer on creating a dashboard, I will start by searching for general guidelines or steps related to dashboard creation. This should give a foundational understanding of the process.

Let's search for "creating a dashboard."
executing search({"query":"creating a dashboard"})...

iteraration number 2...
ASSISTANT: The search for "creating a dashboard" returned useful information on how to add panels and tabs within a dashboard. Here are the key takeaways:

1. **Adding Tabs**: You can organize your panels by adding multiple tabs. This can be done in "Edit" mode by clicking the plus sign and choosing "add Tab."

2. **Adding Panels**: You can add various types of panels including text panels, counters, pie charts, and plots. The process involves entering "Edit" mode, clicking on "Add Panel," and following the configuration prompts.

3. **Python API Option**: If you're using Python, you can utilize the API to manage your dashboar

In [23]:
# The agent conducted research and provided a comprehensive answer. Next, ask it to add this content to the database:

message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [24]:
# Run the loop again:

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 4...
executing add_entry({"filename":"dashboard_creation.mdx","title":"Creating and Managing Dashboards","description":"Steps to create and manage dashboards including adding tabs and panels.","content":"### 1. **Creating Tabs**\n- **Edit Mode**: Enter \"Edit\" mode at the top right of the dashboard.\n- **Add Tab**: Click on the plus sign, and choose \"Add Tab.\" You can create a custom tab by selecting \"empty\" and entering a name.\n- **Templates**: You can also start with pre-built tab templates, which come with predefined panel combinations.\n\n### 2. **Adding Panels**\nYou can add various types of panels to visualize evaluation results:\n\n- **General Steps**:\n  - Enter \"Edit\" mode.\n  - Click \"Add Panel.\"\n  - Configure the panel through prompts and use the preview to ensure correctness.\n  - Save the panel and choose the tab where it belongs.\n\n- **Types of Panels**:\n  - **Text Panels**: Ideal for titles.\n    ```python\n    project.dashboard.add_panel(

In [25]:
#The agent used the add_entry tool to add the summary to the index. Let's verify:

index.docs[-1]
# The agent added the summary to the index.

{'start': 0,
 'content': '### 1. **Creating Tabs**\n- **Edit Mode**: Enter "Edit" mode at the top right of the dashboard.\n- **Add Tab**: Click on the plus sign, and choose "Add Tab." You can create a custom tab by selecting "empty" and entering a name.\n- **Templates**: You can also start with pre-built tab templates, which come with predefined panel combinations.\n\n### 2. **Adding Panels**\nYou can add various types of panels to visualize evaluation results:\n\n- **General Steps**:\n  - Enter "Edit" mode.\n  - Click "Add Panel."\n  - Configure the panel through prompts and use the preview to ensure correctness.\n  - Save the panel and choose the tab where it belongs.\n\n- **Types of Panels**:\n  - **Text Panels**: Ideal for titles.\n    ```python\n    project.dashboard.add_panel(\n        DashboardPanelPlot(\n            title="Dashboard title",\n            size="full",\n            values=[],  # leave empty for text only\n            plot_params={"plot_type": "text"},\n        ),\

Every agent framework runs on the agentic loop. Instead of manually handling each step, we:
 - Loop until the agent stops making tool calls 
 - Handle any number of tools 
 - Let the agent self-correct if it makes mistakes

